# Session 5: Practical Assessment – Advanced Anomaly Detection

**Course:** Machine Learning III (Unsupervised Learning) @Albert School  
**Format:** Groups of 1 to 3 students.  
**Duration:** 3 hours (Due at the end of the session).  
**Grading:** Graded (Low-impact, incentive-based).

### 📖 The Business Scenario
You are the Lead Data Science team for a major manufacturing firm. The company operates expensive, heavy machinery that occasionally suffers from catastrophic failures, halting production and costing **€100,000 per hour** of downtime. 

Your operations team has provided you with telemetry data from these machines (temperatures, torque, tool wear, etc.). Standard rules-based monitoring is no longer sufficient. Your objective is to build an unsupervised anomaly detection pipeline to flag potential machine failures *before* they occur, while minimizing "Alert Fatigue" (False Positives) for the maintenance crew.

### 🎯 Instructions & Deliverables
You must complete this notebook by addressing two distinct perspectives: the **Technical Data Scientist** and the **Business Manager**.

1. **Part 1: Exploratory Data Analysis (EDA) & Cleaning**
   - Investigate features, missing values, and distributions.
   - Preprocess the data (Standardization, handling categorical variables like `Type`).
2. **Part 2: Modeling & Hyperparameter Tuning**
   - Train 4 models: `IsolationForest`, `OneClassSVM`, `LocalOutlierFactor`, and `EllipticEnvelope`.
   - **Rule:** You must tune the trade-off parameters (`contamination`, `nu`, etc.) and justify your choices.
3. **Part 3: Technical Comparison & Visualizations**
   - Use PCA or t-SNE to project the data into 2D/3D.
   - Overlay the anomalies flagged by your models. 
   - Deep Dive: Isolate specific machines flagged by LOF but missed by iForest (or vice versa) and explain *why* based on the algorithm's mathematical assumptions.
4. **Part 4: Managerial Conclusion & Actionable Strategy**
   - **Cost Matrix:** A False Positive costs **€500**. A False Negative costs **€15,000**.
   - Bring back the `Machine failure` labels (hidden during training) and evaluate your models.
   - Conclude: Which model saves the company the most money?


In [ ]:
# ==========================================
# 🚀 INITIALIZATION & DATA LOADING
# Run this cell to get started!
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# 1. Load the AI4I 2020 Predictive Maintenance Dataset directly from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
print("Downloading dataset...")
df_raw = pd.read_csv(url)

print(f"Dataset loaded successfully! Shape: {df_raw.shape}")
display(df_raw.head())


---
## Part 1: Exploratory Data Analysis (EDA) & Cleaning

Dans cette partie nous explorons le dataset AI4I 2020 (10 000 observations capteurs de machines), nous identifions la structure, les distributions, les corrélations, le taux de panne réel, puis nous préparons deux versions de la matrice de features : une scalée (pour LOF, OC-SVM, Elliptic Envelope) et une brute (pour Isolation Forest).


### 1.1 — Structure du dataset

In [ ]:
# Aperçu général : taille, types, valeurs manquantes
print(f"Shape : {df_raw.shape}")
print(f"\nTypes de colonnes :")
print(df_raw.dtypes)
print(f"\nValeurs manquantes par colonne :")
print(df_raw.isna().sum())


In [ ]:
# Statistiques descriptives sur les colonnes numériques
df_raw.describe().T


**Observations :**
- 10 000 lignes, pas de valeurs manquantes (dataset propre).
- `UDI` et `Product ID` sont des identifiants → à drop.
- `Type` est catégorielle (L/M/H) → à encoder.
- 5 sensors numériques principaux : `Air temperature [K]`, `Process temperature [K]`, `Rotational speed [rpm]`, `Torque [Nm]`, `Tool wear [min]`.
- `Machine failure` est la cible (à droper pendant l'entraînement, à utiliser uniquement à la fin).
- `TWF`, `HDF`, `PWF`, `OSF`, `RNF` sont les sous-modes de panne → leakage, à drop aussi.


### 1.2 — Taux de panne réel (uniquement pour cadrer `contamination`)

In [ ]:
# La consigne interdit d'utiliser Machine failure pour entraîner,
# mais on peut s'en servir comme indication pour cadrer le paramètre 'contamination'.
failure_rate = df_raw["Machine failure"].mean()
print(f"Taux de panne réel : {failure_rate:.4f} ({failure_rate*100:.2f} %)")
print(f"Nombre de pannes : {df_raw['Machine failure'].sum()} sur {len(df_raw)}")


**Insight :** ~3.39 % de pannes. On utilisera ce chiffre comme **borne basse** pour `contamination` dans les modèles, puis on testera des valeurs supérieures (le coût asymétrique FP/FN nous poussera à sur-détecter).


### 1.3 — Distribution de la variable catégorielle `Type`

In [ ]:
print(df_raw["Type"].value_counts(normalize=True).round(3))
df_raw["Type"].value_counts().plot(kind="bar", color=["#4C72B0", "#DD8452", "#55A467"])
plt.title("Distribution des types de machines")
plt.ylabel("Nombre de machines")
plt.show()


**Observation :** Trois catégories L (low quality, ~60 %), M (medium, ~30 %), H (high, ~10 %). Distribution déséquilibrée mais c'est cohérent avec un parc industriel réel.


### 1.4 — Distributions des sensors numériques

In [ ]:
sensor_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), sensor_cols):
    ax.hist(df_raw[col], bins=50, color="#4C72B0", edgecolor="black", alpha=0.8)
    ax.set_title(col)
    ax.set_ylabel("Fréquence")

# Skewness pour quantifier l'asymétrie
print("Skewness par sensor :")
print(df_raw[sensor_cols].skew().round(3))

axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()


**Observations :**
- `Air temperature` et `Process temperature` sont quasi-gaussiennes (skewness proche de 0) → **bonne nouvelle pour Elliptic Envelope** qui suppose une distribution gaussienne.
- `Rotational speed` est fortement asymétrique à droite (skew > 1) → présence d'une queue lourde, candidate à des anomalies.
- `Torque` est légèrement asymétrique mais reste bell-shaped.
- `Tool wear` est quasi-uniforme (machines à différents âges d'usure).

→ La distribution non-gaussienne de `Rotational speed` va probablement poser problème à **Elliptic Envelope** mais sera bien gérée par Isolation Forest et LOF.


### 1.5 — Matrice de corrélation

In [ ]:
corr = df_raw[sensor_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Corrélations entre sensors")
plt.tight_layout()
plt.show()


**Observations :**
- `Air temperature` et `Process temperature` sont **fortement corrélées** (~0.88) — physiquement attendu, la température du process suit l'ambiante.
- `Rotational speed` et `Torque` sont **fortement anti-corrélées** (~ -0.88) — relation physique : à puissance ≈ constante, plus la vitesse augmente, plus le couple baisse.
- Les autres paires sont faiblement corrélées.

→ Cette corrélation forte entre 2 paires veut dire que la variance n'est pas isotrope. **C'est exactement ce qu'Elliptic Envelope est censé bien capturer** (ellipsoïde de Mahalanobis), tandis qu'un LOF en distance Euclidienne brute pourrait surévaluer ces axes.


### 1.6 — Preprocessing

Décisions :
1. **Drop des colonnes** : `UDI`, `Product ID` (identifiants), `Machine failure` (cible cachée), `TWF`/`HDF`/`PWF`/`OSF`/`RNF` (sous-modes de panne → leakage).
2. **Encodage de `Type`** : One-Hot encoding (avec `drop_first=True` pour éviter la colinéarité). On préfère One-Hot à un encodage ordinal car la « hiérarchie » L<M<H ne correspond pas forcément à une distance proportionnelle dans l'espace des features.
3. **Standardisation** des numériques avec `StandardScaler` (centrer-réduire).

**Pourquoi le scaling est crucial pour LOF / Elliptic Envelope mais pas pour Isolation Forest :**
- **LOF** repose sur des distances Euclidiennes locales (k plus proches voisins). Si une feature a une variance 1000x plus grande qu'une autre (ex : `Rotational speed [rpm]` ~1500 vs `Torque [Nm]` ~40), elle dominera la distance et écrasera l'information des autres features. Scaler met toutes les features sur la même échelle.
- **Elliptic Envelope** estime la matrice de covariance et utilise la distance de Mahalanobis. Mathématiquement, Mahalanobis est invariante à la mise à l'échelle linéaire — mais en pratique, l'estimation robuste de la covariance (MCD) est plus stable numériquement sur des données centrées-réduites.
- **One-Class SVM** avec kernel RBF utilise une distance euclidienne dans le calcul du noyau → même argument que LOF, scaling indispensable.
- **Isolation Forest** au contraire fait des splits aléatoires axis-aligned (par seuil sur une feature à la fois). Un split à `Torque > 50` produit la même partition quelle que soit l'échelle. → **Invariant à toute transformation monotone par feature**, scaling inutile (mais inoffensif).


In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Drop des colonnes non-features
leakage_cols = ["UDI", "Product ID", "Machine failure",
                "TWF", "HDF", "PWF", "OSF", "RNF"]
df = df_raw.drop(columns=leakage_cols)

# 2. One-Hot encoding de Type (drop_first pour éviter la dummy trap)
df_encoded = pd.get_dummies(df, columns=["Type"], drop_first=True, dtype=float)

print(f"Features après encoding : {list(df_encoded.columns)}")
print(f"Shape : {df_encoded.shape}")

# 3. Deux versions :
#    - X_raw : pour Isolation Forest (invariant à l'échelle)
#    - X_scaled : pour LOF, OC-SVM, Elliptic Envelope (distance-based)
X_raw = df_encoded.values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded)

# y_true conservée à part (à n'utiliser qu'à la Partie 4 !)
y_true = df_raw["Machine failure"].values

print(f"\nX_raw shape    : {X_raw.shape}")
print(f"X_scaled shape : {X_scaled.shape}")
print(f"Moyenne X_scaled (doit ≈ 0) : {X_scaled.mean():.4f}")
print(f"Std X_scaled    (doit ≈ 1) : {X_scaled.std():.4f}")


**Récap Partie 1 :**
- Dataset propre, 10 000 obs, 0 NaN.
- 5 sensors numériques + 1 catégorielle (`Type`).
- Taux de panne réel ≈ 3.39 % → cible pour `contamination`.
- 2 paires de features fortement corrélées (températures ; speed/torque) → covariance non-isotrope.
- Distribution de `Rotational speed` skewed → défi pour les modèles gaussiens.
- 2 matrices de features prêtes : `X_raw` pour iForest, `X_scaled` pour les autres.


---
## Part 2: Modeling & Hyperparameter Tuning
*(Train IsolationForest, OneClassSVM, LocalOutlierFactor, and EllipticEnvelope. Remember to tune your threshold parameters!)*


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope

# Your code here


---
## Part 3: Technical Comparison & Visualizations
*(Use PCA/t-SNE to visualize the flagged anomalies. Find an anomaly caught by one model but missed by another and explain why.)*


In [ ]:
from sklearn.decomposition import PCA

# Your code here


---
## Part 4: Managerial Conclusion & Business Strategy
*(Bring back `y_true`. Calculate the number of False Positives and False Negatives for each model. Apply the cost matrix. Which model wins?)*


In [ ]:
# Business Cost Matrix
COST_FP = 500     # False Positive: Wasted technician check
COST_FN = 15000   # False Negative: Catastrophic machine breakdown

# Your code here
